In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

project_root = Path.cwd().parents[2]

feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_3_other_skill_history"
    / "v3.2"
)

prediction_dir = (
    project_root
    / "data"
    / "processed"
    / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Feature directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_3_other_skill_history\v3.2
Prediction directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:
v32_fold_datasets = []

for fold_number in range(1, 6):

    train_path = feature_dir / f"fold_{fold_number}_train.csv"
    val_path = feature_dir / f"fold_{fold_number}_validation.csv"

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    v32_fold_datasets.append((train_df, val_df))

    print(
        f"Fold {fold_number}: "
        f"Train = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

print("\nAll V3.2 datasets loaded successfully!")

Fold 1: Train = (177324, 4), Validation = (42659, 4)
Fold 2: Train = (176273, 4), Validation = (43710, 4)
Fold 3: Train = (170186, 4), Validation = (49797, 4)
Fold 4: Train = (178311, 4), Validation = (41672, 4)
Fold 5: Train = (177838, 4), Validation = (42145, 4)

All V3.2 datasets loaded successfully!


In [3]:
feature_column = "weighted_other_skill_smoothed_accuracy"
target_column = "correct"

for fold_number, (train_df, val_df) in enumerate(
    v32_fold_datasets, start=1
):
    print(f"\n========== Fold {fold_number} ==========")

    print("Training missing values:", train_df.isna().sum().sum())
    print("Validation missing values:", val_df.isna().sum().sum())

    print(
        "Training feature range:",
        train_df[feature_column].min(),
        train_df[feature_column].max()
    )

    print(
        "Validation feature range:",
        val_df[feature_column].min(),
        val_df[feature_column].max()
    )

    print("\nTraining target distribution:")
    print(train_df[target_column].value_counts(normalize=True))

    print("\nValidation target distribution:")
    print(val_df[target_column].value_counts(normalize=True))


========== Fold 1 ==========
Training missing values: 0
Validation missing values: 0
Training feature range: 0.0337005285742449 1.0
Validation feature range: 0.0615807097922183 0.9350959020125996

Training target distribution:
correct
1    0.660136
0    0.339864
Name: proportion, dtype: float64

Validation target distribution:
correct
1    0.652336
0    0.347664
Name: proportion, dtype: float64

========== Fold 2 ==========
Training missing values: 0
Validation missing values: 0
Training feature range: 0.0334387165788468 1.0
Validation feature range: 0.076319245509074 0.932089708656466

Training target distribution:
correct
1    0.654496
0    0.345504
Name: proportion, dtype: float64

Validation target distribution:
correct
1    0.675269
0    0.324731
Name: proportion, dtype: float64

========== Fold 3 ==========
Training missing values: 0
Validation missing values: 0
Training feature range: 0.0337779392543559 1.0
Validation feature range: 0.0621262278267067 0.9264554713433324

Traini

In [4]:
feature_column = ["weighted_other_skill_smoothed_accuracy"]
target_column = "correct"

v32_models = []
v32_fold_results = []

for fold_number, (train_df, val_df) in enumerate(
    v32_fold_datasets, start=1
):
    print(f"\nTraining Fold {fold_number}...")

    X_train = train_df[feature_column]
    y_train = train_df[target_column]

    X_val = val_df[feature_column]
    y_val = val_df[target_column]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    v32_models.append(model)

    v32_fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier
    })

    print(f"Log Loss: {fold_log_loss:.4f}")
    print(f"Brier Score: {fold_brier:.4f}")

print("\nAll five V3.2 models trained successfully!")


Training Fold 1...
Log Loss: 0.6161
Brier Score: 0.2134

Training Fold 2...
Log Loss: 0.6112
Brier Score: 0.2109

Training Fold 3...
Log Loss: 0.6290
Brier Score: 0.2192

Training Fold 4...
Log Loss: 0.6173
Brier Score: 0.2137

Training Fold 5...
Log Loss: 0.6092
Brier Score: 0.2101

All five V3.2 models trained successfully!


In [5]:
v32_predictions = []

for fold_number, (train_df, val_df) in enumerate(
    v32_fold_datasets, start=1
):
    # Retrieve the already-trained model for this fold
    model = v32_models[fold_number - 1]

    # Predict probabilities on the validation fold
    X_val = val_df[["weighted_other_skill_smoothed_accuracy"]]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual labels and predicted probabilities
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold_number,
        "y_true": val_df["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V3.2"
    })

    v32_predictions.append(fold_predictions)

    print(
        f"Fold {fold_number}: "
        f"{len(fold_predictions)} predictions generated."
    )

# Combine predictions from all five validation folds
v32_predictions_df = pd.concat(
    v32_predictions,
    ignore_index=True
)

# Save predictions
prediction_path = prediction_dir / "v3_2_predictions.csv"

v32_predictions_df.to_csv(
    prediction_path,
    index=False
)

print("\nV3.2 predictions saved successfully!")
print("Shape:", v32_predictions_df.shape)
print("Location:", prediction_path)

Fold 1: 42659 predictions generated.
Fold 2: 43710 predictions generated.
Fold 3: 49797 predictions generated.
Fold 4: 41672 predictions generated.
Fold 5: 42145 predictions generated.

V3.2 predictions saved successfully!
Shape: (219983, 6)
Location: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v3_2_predictions.csv


In [6]:
saved_predictions = pd.read_csv(prediction_path)

print("Shape:", saved_predictions.shape)

print("\nMissing values:")
print(saved_predictions.isna().sum())

print("\nFold distribution:")
print(saved_predictions["fold"].value_counts().sort_index())

print("\nActual target distribution:")
print(saved_predictions["y_true"].value_counts())

print("\nPredicted probability range:")
print(
    saved_predictions["y_pred_proba"].min(),
    saved_predictions["y_pred_proba"].max()
)

print("\nFirst five predictions:")
print(saved_predictions.head())

Shape: (219983, 6)

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Actual target distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64

Predicted probability range:
0.1616308329644777 0.8674175514155165

First five predictions:
   user_id  order_id  fold  y_true  y_pred_proba model
0    21825  21616385     1       0      0.716267  V3.2
1    21825  21616478     1       1      0.716267  V3.2
2    21825  21616677     1       1      0.716267  V3.2
3    21825  21616919     1       1      0.716267  V3.2
4    21825  21671620     1       0      0.715961  V3.2
